# 전국 화재 데이터로 화재 원인 예측하기

- 데이터: **소방청_화재발생 정보** — 공공데이터포털(data.go.kr), 이용허락범위 제한 없음
  - 2020년 1월 ~ 2024년 12월 전국에서 발생한 **실제 화재 191,510건**
  - 발생 일시, 지역, 화재 유형, 장소(대·중·소분류), 발화 원인, 인명·재산 피해가 화재 1건마다 기록됨
  - 원본(cp949 CSV)을 `make_dataset.py`로 영어 컬럼명 + 엑셀에서 열리는 CSV로 바꿔 사용
  - 분석에 쓰지 않는 3개 컬럼(시군구, 부동산·동산 피해 세부)은 빼서 GitHub 업로드 제한(25MB) 아래로 맞춤 (행은 그대로)
- 목표: 화재가 **언제(월·시간·요일), 어디서(지역·장소), 어떤 종류로** 났는지만으로 **발화 원인**(부주의 / 전기적 요인 / 기계적 요인 / 기타)을 예측하고,
  시기·장소별로 어떤 원인을 조심해야 하는지 해석
- 활용: 원인 조사가 끝나기 전 **우선 점검할 방향** 제시, 원인이 '미상'으로 남은 화재의 원인 추정, 시기·장소별 **맞춤 예방 홍보**

In [ ]:
# ── 공통 설정 ──
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 한글 폰트 설정 (윈도우 기본 '맑은 고딕'), 마이너스 기호 깨짐 방지
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 결과 재현을 위해 난수 시드를 42로 고정
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# 경로 설정: 데이터는 dataset 폴더, 발표용 그림은 outputs 폴더에 저장
DATA_PATH = os.path.join('dataset', 'fire_2020_2024.csv')
OUT_DIR = 'outputs'
os.makedirs(OUT_DIR, exist_ok=True)

# 그래프에서 원인별 색을 항상 같게 쓰기 위한 설정
CLASSES = ['부주의', '전기적 요인', '기계적 요인', '기타']
COLORS = {'부주의': '#f58518', '전기적 요인': '#4c78a8', '기계적 요인': '#54a24b', '기타': '#9aa5b1'}

## 1. 데이터 확인

In [ ]:
# 파일이 없으면 여기서 멈추도록 에러를 발생시킴
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f'{DATA_PATH} 파일이 없습니다. 소방청 원본 CSV를 dataset/raw 폴더에 넣고 make_dataset.py를 먼저 실행해 주세요.')

# utf-8-sig: 엑셀용으로 저장한 파일 맨 앞의 표시(BOM)를 빼고 읽음
df = pd.read_csv(DATA_PATH, encoding='utf-8-sig', parse_dates=['datetime'])
print('데이터 크기(행, 열):', df.shape)
print('기간:', df['datetime'].min(), '~', df['datetime'].max())
df.head()

In [ ]:
# 컬럼 자료형과 결측치, 중복 확인
df.info()
print()
print('결측치 개수 합계:', df.isnull().sum().sum())
print('완전히 같은 행:', df.duplicated().sum())

### 1-2. 컬럼 설명

| 컬럼 | 원본 항목명 | 의미 |
|---|---|---|
| datetime | 화재발생년원일 | 발생 일시 (분 단위) |
| sido | 시도 | 발생 지역 (17개 시도) |
| fire_type | 화재유형 | 건축·구조물, 자동차·철도차량, 임야, 기타(쓰레기 화재 등) 등 6종 |
| **cause** | 발화요인대분류 | **예측 대상**: 부주의, 전기적 요인, 기계적 요인, 방화, 미상 등 12종 |
| cause_detail | 발화요인소분류 | 세부 원인 (담배꽁초, 음식물 조리중, 과부하 등) — **원인 조사 결과이므로 입력에 쓰지 않음** |
| casualties / deaths / injuries | 인명피해 소계 / 사망 / 부상 | 명 |
| damage | 재산피해 소계 | 천원 |
| place_1 / place_2 / place_3 | 장소 대·중·소분류 | 예: 생활서비스 > 음식점 > 치킨·족발 |

In [ ]:
# 범주형 컬럼별 종류 수
for col in ['sido', 'fire_type', 'cause', 'place_1', 'place_2', 'place_3']:
    print(f'{col}: {df[col].nunique()}종')
print()
print(df['cause'].value_counts().to_string())

## 2. 탐색적 데이터 분석(EDA)

### 2-1. 원인 그룹 정하기
12종 중 일부는 건수가 매우 적어서(가스누출 707건, 제품결함 801건 등) 그대로 예측하기 어렵습니다.
건수가 많은 3개(부주의·전기적·기계적 요인)는 그대로 두고, 나머지는 **'기타'**로 묶습니다.
**'미상'**(원인을 밝히지 못한 화재)은 정답이 없으므로 학습에서 빼고, 5-2에서 모델로 원인을 추정해 봅니다.

In [ ]:
def group_cause(c):
    return c if c in ['부주의', '전기적 요인', '기계적 요인', '미상'] else '기타'

df['cause_group'] = df['cause'].apply(group_cause)
print(df['cause_group'].value_counts().to_string())
print()
print('기타에 묶인 원인:', sorted(df.loc[df['cause_group'] == '기타', 'cause'].unique()))

# EDA·모델링 모두 원인을 아는 화재만 사용
known = df[df['cause_group'] != '미상'].copy()
known['month'] = known['datetime'].dt.month
known['hour'] = known['datetime'].dt.hour
print(f'\n원인을 아는 화재: {len(known):,}건 / 원인 미상: {(df["cause_group"] == "미상").sum():,}건 ({(df["cause_group"] == "미상").mean():.1%})')

### 2-2. 계절(월)에 따라 원인이 달라질까?

In [ ]:
by_month = pd.crosstab(known['month'], known['cause_group'], normalize='index')[CLASSES] * 100
counts = known.groupby('month').size()

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
axes[0].bar(counts.index, counts.values, color='#bbbbbb')
axes[0].set_title('월별 화재 건수 (원인 미상 제외, 5년 합계)')
axes[0].set_xlabel('월')
axes[0].set_xticks(range(1, 13))
for c in CLASSES:
    axes[1].plot(by_month.index, by_month[c], marker='o', color=COLORS[c], label=c)
axes[1].set_title('월별 원인 비율')
axes[1].set_xlabel('월')
axes[1].set_ylabel('비율(%)')
axes[1].set_xticks(range(1, 13))
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, '01_월별_화재건수_원인비율.png'), dpi=200, bbox_inches='tight')
plt.show()

print(by_month.round(1).to_string())

**관찰:** 계절에 따라 원인 구성이 뚜렷하게 바뀝니다. **7~8월에는 전기적 요인이 42~43%로 부주의(약 35%)보다 많아지고**, **3~4월에는 부주의가 62~63%로 가장 높습니다.**
여름에는 에어컨·선풍기 등 전기 사용이 늘고, 봄에는 건조한 날씨에 쓰레기·논밭두렁 태우기와 담배꽁초 화재가 많기 때문으로 보입니다. 월은 원인을 구분하는 데 쓸모 있는 정보입니다.

### 2-3. 하루 중 시간대에 따라 원인이 달라질까?

In [ ]:
by_hour = pd.crosstab(known['hour'], known['cause_group'], normalize='index')[CLASSES] * 100
fig, ax = plt.subplots(figsize=(12, 4))
for c in CLASSES:
    ax.plot(by_hour.index, by_hour[c], marker='o', markersize=4, color=COLORS[c], label=c)
ax.set_xticks(range(0, 24))
ax.set_xlabel('발생 시각(시)')
ax.set_ylabel('비율(%)')
ax.set_title('시간대별 원인 비율')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, '02_시간대별_원인비율.png'), dpi=200, bbox_inches='tight')
plt.show()

print(by_hour.loc[[3, 9, 14, 19, 23]].round(1).to_string())

**관찰:** 오후 2시에는 부주의가 61.5%로 가장 높지만, **새벽 3시에는 전기적 요인(37.6%)이 부주의(36.9%)와 비슷해지고 기타(15.3%)도 늘어납니다.**
사람이 활동하는 낮에는 부주의 화재가 많고, 사람이 잠든 새벽에는 전기 설비나 방화 등 사람의 행동과 무관한 원인의 비중이 커집니다.

### 2-4. 장소에 따라 원인이 달라질까?

In [ ]:
# 장소 중분류 중 화재가 많은 15곳의 원인 구성
top_places = known['place_2'].value_counts().head(15).index
by_place = pd.crosstab(known['place_2'], known['cause_group'], normalize='index').loc[top_places, CLASSES] * 100

fig, ax = plt.subplots(figsize=(11, 6))
left = np.zeros(len(by_place))
for c in CLASSES:
    ax.barh(by_place.index, by_place[c], left=left, color=COLORS[c], label=c)
    left += by_place[c].values
ax.invert_yaxis()
ax.set_xlabel('비율(%)')
ax.set_title('장소(중분류)별 원인 구성 — 화재가 많은 15곳')
ax.legend(loc='lower right')
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, '03_장소별_원인구성.png'), dpi=200, bbox_inches='tight')
plt.show()

print(by_place.round(1).to_string())

**관찰:** 장소에 따라 원인이 크게 다릅니다.
- **들불(95.4%)·산불(91.3%)·야외(77.8%)**는 거의 부주의
- **자동차**는 기계적 요인(36.6%)이 가장 많음
- **동식물시설(55.7%)·일상서비스(44.2%)·판매시설(43.9%)**은 전기적 요인이 가장 많음
- **공장시설**은 부주의·전기·기계가 약 30%씩 고르게 섞여 있어 장소만으로는 원인을 가리기 어려움
장소가 원인 예측의 핵심 정보가 될 것으로 예상됩니다.

### 2-5. 세부 원인 TOP 15

In [ ]:
top_detail = known['cause_detail'].value_counts().head(15)
group_of = known.drop_duplicates('cause_detail').set_index('cause_detail')['cause_group']

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.barh(top_detail.index, top_detail.values, color=[COLORS[group_of[d]] for d in top_detail.index])
ax.invert_yaxis()
ax.set_xlabel('건수 (2020~2024)')
ax.set_title('세부 원인 TOP 15 (색 = 원인 그룹)')
for i, v in enumerate(top_detail.values):
    ax.text(v + 200, i, f'{v:,}', va='center')
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, '04_세부원인_TOP15.png'), dpi=200, bbox_inches='tight')
plt.show()

print(f'담배꽁초 비중: 전체 화재의 {(df["cause_detail"] == "담배꽁초").mean():.1%}')

**관찰:** **담배꽁초가 29,030건으로 1위이며, 전체 화재의 15.2%**를 차지합니다. 그 다음은 미확인단락(전기), 음식물 조리중, 과열·과부하(기계), 불씨·불꽃 방치, 쓰레기 소각 순입니다.
상위권 대부분이 **사람의 작은 부주의**로 생기는 화재라서 예방 홍보로 줄일 수 있는 여지가 큽니다.

### 2-6. 어떤 원인의 화재가 더 위험할까? (인명·재산 피해)

In [ ]:
harm = known.groupby('cause_group').agg(
    건수=('casualties', 'size'),
    인명피해_발생률=('casualties', lambda s: (s > 0).mean() * 100),
    사망자=('deaths', 'sum'),
    재산피해_중앙값_만원=('damage', lambda s: s.median() / 10),
    재산피해_평균_만원=('damage', lambda s: s.mean() / 10),
).loc[CLASSES]
print(harm.round(1).to_string())

# 재산피해는 0원부터 수천억 원까지 퍼져 있어 로그 눈금으로 비교
fig, ax = plt.subplots(figsize=(10, 4))
for c in CLASSES:
    d = known.loc[(known['cause_group'] == c) & (known['damage'] > 0), 'damage'] * 1000
    sns.kdeplot(np.log10(d), ax=ax, color=COLORS[c], label=c)
ax.set_xticks(range(3, 11))
ax.set_xticklabels(['1천원', '1만원', '10만원', '100만원', '1천만원', '1억원', '10억원', '100억원'])
ax.set_xlabel('재산피해 (로그 눈금)')
ax.set_title('원인별 재산피해 분포')
ax.legend()
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, '05_원인별_재산피해분포.png'), dpi=200, bbox_inches='tight')
plt.show()

**관찰:** 건수는 부주의가 가장 많지만 **피해 정도는 원인마다 다릅니다.**
- **기타**(방화·가스누출·교통사고 등)는 인명피해 발생률이 11.0%로 가장 높고, 사망자도 469명으로 가장 많습니다.
- **전기적 요인**은 재산피해 중앙값이 100.9만원으로 **부주의(21.6만원)의 약 4.7배**입니다. 건물 안에서 늦게 발견되는 경우가 많기 때문으로 보입니다.
- 부주의 화재는 건수가 많은 대신 쓰레기·야외 화재처럼 피해가 작은 경우가 많습니다.

## 3. 전처리

In [ ]:
# 1) 입력 변수의 원재료: 발생 시점(월·시간·요일)과 장소·지역·화재 유형
#    - 세부 원인(cause_detail)과 피해 규모는 조사가 끝난 뒤에 알 수 있는 정보라 입력에서 제외 (정답 누설 방지)
known['weekday'] = known['datetime'].dt.day_of_week   # 0 = 월요일
known['year'] = known['datetime'].dt.year

# 2) 장소 소분류는 364종이라, 학습 기간(2020~2023)에 30건 미만인 드문 장소는 '기타(드문 장소)'로 묶음
place3_counts = known.loc[known['year'] < 2024, 'place_3'].value_counts()
KEEP_PLACE3 = sorted(place3_counts[place3_counts >= 30].index)
print(f'장소 소분류 {known["place_3"].nunique()}종 중 {len(KEEP_PLACE3)}종 유지, 나머지는 기타로 묶음')

def prepare(d):
    out = d[['month', 'hour', 'weekday', 'sido', 'fire_type', 'place_1', 'place_2']].copy()
    out['place_3'] = d['place_3'].where(d['place_3'].isin(KEEP_PLACE3), '기타(드문 장소)')
    return out

# 3) 모든 입력을 원-핫 인코딩 (월·시간·요일도 '숫자 크기'가 아니라 '구분'으로 다룸)
#    범주 목록을 고정해 두어야 앱에서 한 건만 넣어도 같은 컬럼이 만들어짐
raw_inputs = prepare(known)
CATEGORIES = {col: sorted(raw_inputs[col].unique()) for col in raw_inputs.columns}

def encode(d):
    parts = [pd.get_dummies(pd.Categorical(d[col], categories=cats), prefix=col, dtype=int).set_axis(d.index)
             for col, cats in CATEGORIES.items()]
    return pd.concat(parts, axis=1)

X_all = encode(raw_inputs)
y_all = known['cause_group'].map({c: i for i, c in enumerate(CLASSES)})
print('입력 컬럼 수:', X_all.shape[1])
print('예측 대상:', dict(enumerate(CLASSES)))

## 4. 모델링

### 4-1. 학습/테스트 분할: 시간 순서대로 (2020~2023 학습 → 2024 테스트)
무작위로 나누면 같은 시기의 화재가 학습·테스트에 섞여 **미래를 미리 본** 셈이 됩니다.
실제로는 과거 기록으로 앞으로의 화재를 예측해야 하므로, **2020~2023년으로 학습하고 2024년으로 평가**합니다.

In [ ]:
train_mask = (known['year'] < 2024).values
X_train, X_test = X_all[train_mask], X_all[~train_mask]
y_train, y_test = y_all[train_mask], y_all[~train_mask]
print(f'학습(2020~2023): {len(X_train):,}건 / 테스트(2024): {len(X_test):,}건')
print('테스트 데이터 원인 비율:')
print(y_test.map(dict(enumerate(CLASSES))).value_counts(normalize=True).round(3).to_string())

### 4-2. 평가 지표
- **정확도**: 원인을 정확히 맞힌 비율. 단, '부주의'가 절반이라 **모두 부주의라고 찍어도 약 49%**가 나옵니다.
- **매크로 F1**: 4개 원인 각각의 F1을 똑같은 비중으로 평균 → 건수가 적은 원인도 잘 맞히는지 봄
- **상위 2개 적중률**: 모델이 가장 가능성 높다고 본 원인 2개 안에 정답이 있는 비율 → '우선 점검할 방향'으로 쓸 때의 쓸모

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, top_k_accuracy_score, classification_report, confusion_matrix

def evaluate(name, y_true, proba):
    pred = proba.argmax(axis=1)
    return {'모델': name,
            '정확도': accuracy_score(y_true, pred),
            '매크로 F1': f1_score(y_true, pred, average='macro'),
            '상위 2개 적중률': top_k_accuracy_score(y_true, proba, k=2, labels=range(len(CLASSES)))}

### 4-3. 모델 4개 학습: 최빈값 찍기(기준), 로지스틱 회귀, 랜덤포레스트, XGBoost

In [ ]:
candidates = [
    ('0 최빈값 찍기 (기준)', 'dummy', DummyClassifier(strategy='prior')),
    ('① 로지스틱 회귀', 'lr', LogisticRegression(max_iter=1000)),
    ('② 랜덤포레스트', 'rf', RandomForestClassifier(n_estimators=200, min_samples_leaf=5,
                                                  random_state=RANDOM_STATE, n_jobs=-1)),
    ('③ XGBoost', 'xgb', XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, tree_method='hist',
                                       random_state=RANDOM_STATE, n_jobs=-1)),
]
models, proba, rows, train_rows = {}, {}, [], []
for name, key, m in candidates:
    m.fit(X_train, y_train)
    models[key] = m
    proba[key] = m.predict_proba(X_test)
    rows.append(evaluate(name, y_test, proba[key]))
    train_rows.append(evaluate(name, y_train, m.predict_proba(X_train)))
result_df = pd.DataFrame(rows).set_index('모델')
train_df = pd.DataFrame(train_rows).set_index('모델')
result_df['학습 정확도'] = train_df['정확도']
result_df.round(4)

**관찰:** 학습한 세 모델의 성능은 거의 같습니다(정확도 0.576~0.578, 매크로 F1 0.416~0.424). 최빈값 찍기와 비교하면 다음과 같습니다.
- **정확도:** 0.490 → 0.578 (**+8.9%p**)
- **매크로 F1:** 0.164 → 0.424
- **상위 2개 적중률:** 0.796 → 0.843 (부주의·전기 두 개만 말해도 79.6%라 차이가 크지 않음)
학습 정확도(0.60~0.64)와 테스트 정확도의 차이가 작아 과적합은 거의 없습니다.
**장소·시간 정보만으로는 원인을 절반 남짓 맞히는 수준**이며, 같은 장소·시간이라도 원인이 다양하다는 화재의 특성상 한계가 분명합니다.

In [ ]:
# 테스트 매크로 F1이 가장 높은 모델을 최종 모델로 선택 (정확도는 '부주의'만 잘 맞혀도 높게 나오므로)
key_of = {name: key for name, key, _ in candidates}
best_name = result_df['매크로 F1'].idxmax()
best_key = key_of[best_name]
best_model, proba_best = models[best_key], proba[best_key]
print('최종 모델:', best_name)

fig, axes = plt.subplots(1, 3, figsize=(16, 3.8), sharey=True)
colors = ['#cccccc', '#9aa5b1', '#4c78a8', '#4c78a8']
for ax, col in zip(axes, ['정확도', '매크로 F1', '상위 2개 적중률']):
    ax.barh(result_df.index, result_df[col], color=colors)
    for i, v in enumerate(result_df[col]):
        ax.text(v + 0.01, i, f'{v:.3f}', va='center')
    ax.set_xlim(0, 1)
    ax.set_title(f'테스트(2024) {col}')
axes[0].invert_yaxis()
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, '06_모델별_성능비교.png'), dpi=200, bbox_inches='tight')
plt.show()

### 4-4. 원인별로 얼마나 맞히나? (최종 모델)

In [ ]:
pred_best = proba_best.argmax(axis=1)
print(classification_report(y_test, pred_best, target_names=CLASSES, digits=3))

# 혼동행렬: 실제 원인별로 모델이 무엇이라고 답했는지 (행 기준 비율)
cm = confusion_matrix(y_test, pred_best, normalize='true') * 100
fig, ax = plt.subplots(figsize=(6.5, 5))
sns.heatmap(cm, annot=True, fmt='.1f', cmap='Blues', cbar=False, ax=ax, xticklabels=CLASSES, yticklabels=CLASSES)
ax.set_xlabel('모델이 예측한 원인')
ax.set_ylabel('실제 원인')
ax.set_title(f'혼동행렬 (행 기준 %, {best_name[2:]}, 2024년)')
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, '07_혼동행렬.png'), dpi=200, bbox_inches='tight')
plt.show()

**관찰:** 원인별 재현율(실제 그 원인인 화재 중 맞힌 비율)은 **부주의 84.9%, 전기적 요인 37.1%, 기계적 요인 36.8%, 기타 8.8%**입니다.
모델은 확실하지 않으면 가장 흔한 '부주의'로 답하는 경향이 있습니다. '기타'는 방화·가스·교통사고 등 성격이 다른 원인을 묶은 그룹이라 공통된 패턴이 약해서 거의 맞히지 못합니다.

### 4-5. 연도를 바꿔 가며 검증 (시간 순서 교차검증)
2024년 한 해만 보면 우연일 수 있으므로, **"그 전 해까지로 학습 → 다음 해 예측"**을 2022·2023·2024년에 반복합니다.

In [ ]:
from sklearn.base import clone

years = known['year'].values
cv_rows = []
for test_year in [2022, 2023, 2024]:
    tr, te = years < test_year, years == test_year
    for name, key, m in candidates:
        mm = clone(m).fit(X_all[tr], y_all[tr])
        r = evaluate(name, y_all[te], mm.predict_proba(X_all[te]))
        r['테스트 연도'] = test_year
        cv_rows.append(r)
cv_df = pd.DataFrame(cv_rows).pivot_table(index='모델', columns='테스트 연도', values=['정확도', '매크로 F1'])
cv_df.round(3)

**관찰:** 세 해 모두 같은 순서와 비슷한 수준이 나와 **결과가 특정 연도의 우연이 아닙니다.** XGBoost의 매크로 F1은 0.417 / 0.424 / 0.424로 안정적입니다.
흥미롭게도 부주의 비율이 해마다 줄어서(최빈값 정확도 0.540 → 0.514 → 0.490) 찍기 기준은 낮아지는데, **모델과 찍기의 정확도 차이는 6.3%p → 8.1%p → 8.8%p로 커집니다.**

### 4-6. 모델 저장 (joblib)

In [ ]:
import joblib

os.makedirs('model', exist_ok=True)
MODEL_PATH = os.path.join('model', 'fire_cause_model.pkl')
# 앱에서 같은 인코딩을 하도록 범주 목록·장소 목록·원인 이름을 모델과 함께 저장
joblib.dump({'model': best_model, 'classes': CLASSES, 'categories': CATEGORIES,
             'keep_place3': KEEP_PLACE3, 'columns': list(X_all.columns)}, MODEL_PATH)

loaded = joblib.load(MODEL_PATH)
same = np.allclose(loaded['model'].predict_proba(X_test), proba_best)
print('저장 모델:', best_name)
print('저장 파일:', MODEL_PATH, f'({os.path.getsize(MODEL_PATH) / 1024 / 1024:.1f} MB)')
print('불러온 모델의 예측이 같은가:', same)

## 5. 해석

### 5-1. 무엇이 원인 예측에 중요할까? (순열 중요도)
2024년 테스트 데이터에서 **원래 변수 하나를 뒤섞은 뒤** 다시 인코딩해 예측했을 때 매크로 F1이 얼마나 떨어지는지 봅니다.
장소 대·중·소분류는 서로 얽혀 있어 '장소' 하나로 묶어서 섞습니다.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
test_raw = raw_inputs[~train_mask]
base_f1 = f1_score(y_test, pred_best, average='macro')
groups = {'장소(대·중·소)': ['place_1', 'place_2', 'place_3'], '화재 유형': ['fire_type'], '월': ['month'],
          '시간': ['hour'], '요일': ['weekday'], '시도': ['sido']}
perm_rows = {}
for gname, cols in groups.items():
    drops = []
    for _ in range(5):   # 5번 반복해 평균 (우연한 결과를 줄이기 위해)
        shuffled = test_raw.copy()
        shuffled[cols] = shuffled[cols].values[rng.permutation(len(shuffled))]
        drops.append(base_f1 - f1_score(y_test, best_model.predict(encode(shuffled)), average='macro'))
    perm_rows[gname] = {'평균 F1 감소': np.mean(drops), '표준편차': np.std(drops)}
perm_df = pd.DataFrame(perm_rows).T.sort_values('평균 F1 감소')

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(perm_df.index, perm_df['평균 F1 감소'], xerr=perm_df['표준편차'], color='#f58518')
ax.set_title(f'{best_name[2:]} 순열 중요도 (2024년 테스트)')
ax.set_xlabel('변수를 섞었을 때 매크로 F1 감소량 (클수록 중요)')
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, '08_순열중요도.png'), dpi=200, bbox_inches='tight')
plt.show()

print(perm_df.sort_values('평균 F1 감소', ascending=False).round(4).to_string())

**관찰:** **화재 유형(0.102)과 장소(0.075)가 압도적으로 중요**하고, 그 다음이 시간(0.025)·월(0.016)·시도(0.015)입니다. 요일은 거의 영향이 없습니다(0.0004).
'무엇이 어디서 탔는가'가 원인을 가장 잘 알려 주고, '언제'는 보조 정보 역할을 합니다.

### 5-2. 원인 '미상' 화재의 원인 추정
원인을 끝내 밝히지 못한 화재에 모델을 적용해, 어떤 원인이었을 가능성이 높은지 추정합니다.

In [ ]:
unknown = df[df['cause_group'] == '미상'].copy()
unknown['month'] = unknown['datetime'].dt.month
unknown['hour'] = unknown['datetime'].dt.hour
unknown['weekday'] = unknown['datetime'].dt.day_of_week
unk_proba = best_model.predict_proba(encode(prepare(unknown)))

# 확률의 평균 = 미상 화재 전체에서 각 원인이 차지할 것으로 추정되는 비율
compare = pd.DataFrame({
    '원인을 아는 화재(실제 비율)': known['cause_group'].value_counts(normalize=True).reindex(CLASSES) * 100,
    '원인 미상 화재(모델 추정 비율)': pd.Series(unk_proba.mean(axis=0) * 100, index=CLASSES),
})
ax = compare.plot.bar(figsize=(9, 4), color=['#bbbbbb', '#e45756'], rot=0)
ax.set_ylabel('비율(%)')
ax.set_title(f'원인 미상 화재 {len(unknown):,}건의 추정 원인 구성')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, '09_원인미상_화재_추정.png'), dpi=200, bbox_inches='tight')
plt.show()

print(compare.round(1).to_string())
print()
print('원인 미상 화재의 화재 유형 구성 vs 전체:')
print(pd.DataFrame({'미상': unknown['fire_type'].value_counts(normalize=True),
                    '원인 아는 화재': known['fire_type'].value_counts(normalize=True)}).round(3).to_string())

**관찰:** 원인 미상 화재 16,508건에 모델을 적용하면 **부주의 46.3%, 전기적 요인 31.2%, 기계적 요인 12.9%, 기타 9.7%**로 추정됩니다. 원인을 아는 화재보다 **부주의는 적고 전기·기계적 요인은 많습니다.**
미상 화재는 건물(71.9% vs 64.0%)과 자동차(15.4% vs 11.9%) 화재가 많고 쓰레기 화재(8.5% vs 20.0%)가 적기 때문입니다. 불에 타 증거가 사라지기 쉬운 **건물·차량 화재일수록 원인을 밝히기 어렵다**는 해석이 가능합니다.
(단, '미상 화재도 원인을 아는 화재와 비슷한 패턴을 따른다'는 가정 위의 추정입니다.)

### 5-3. 상황별 예측 예시

In [ ]:
def scenario(month, hour, weekday, sido, fire_type, place_1, place_2, place_3):
    row = pd.DataFrame([{'month': month, 'hour': hour, 'weekday': weekday, 'sido': sido, 'fire_type': fire_type,
                         'place_1': place_1, 'place_2': place_2, 'place_3': place_3}])
    return pd.Series(best_model.predict_proba(encode(prepare(row)))[0] * 100, index=CLASSES).round(1)

examples = {
    '1월 저녁 7시, 서울 치킨·족발집': (1, 19, 4, '서울특별시', '건축,구조물', '생활서비스', '음식점', '치킨,족발'),
    '8월 오후 3시, 경기 아파트': (8, 15, 2, '경기도', '건축,구조물', '주거', '공동주택', '아파트'),
    '4월 오후 2시, 경북 논밭두렁': (4, 14, 5, '경상북도', '임야', '임야', '들불', '논밭두렁'),
    '7월 오후 1시, 경기 승용차': (7, 13, 1, '경기도', '자동차,철도차량', '자동차,철도차량', '자동차', '승용자동차'),
}
pd.DataFrame({k: scenario(*v) for k, v in examples.items()}).T

**관찰:** - **1월 저녁 7시 치킨집:** 부주의 54.1% (조리 중 부주의)
- **8월 오후 3시 아파트:** 전기적 요인 53.5%로 1위 (여름철 전기 사용)
- **4월 오후 2시 논밭두렁:** 부주의 96.9% (논밭 태우기)
- **7월 오후 1시 승용차:** 기계적 요인 37.3%, 전기적 요인 32.7%로 비슷
같은 화재 유형이라도 장소와 계절에 따라 우선 의심할 원인이 달라집니다.

## 6. 결론

### 6-1. 결론 요약

**1. 성능 (2024년 테스트 34,561건, 2020~2023년으로 학습)**

| 모델 | 정확도 | 매크로 F1 | 상위 2개 적중률 |
|---|---|---|---|
| 최빈값 찍기 (기준) | 0.490 | 0.164 | 0.796 |
| 로지스틱 회귀 | 0.576 | 0.422 | 0.840 |
| 랜덤포레스트 | 0.576 | 0.416 | 0.842 |
| **XGBoost (최종)** | **0.578** | **0.424** | **0.843** |

- 장소·시간·화재 유형만으로 원인을 **기준보다 8.9%p 더 잘 맞히고**, 매크로 F1은 0.164 → 0.424로 높였습니다.
- 2022·2023·2024년 모두 비슷한 성능이 나와 결과가 안정적입니다. 세 모델의 차이는 매우 작습니다.
- 다만 **정확도 58% 수준으로 원인을 확정할 수는 없으며**, '우선 의심할 원인'을 알려 주는 보조 도구로 보는 것이 맞습니다.

**2. 알게 된 화재 원인 패턴**
1. **장소와 화재 유형이 가장 중요**: 들불·산불·야외는 90% 안팎이 부주의, 자동차는 기계적 요인, 동식물시설·판매시설은 전기적 요인이 1위입니다.
2. **계절**: 7~8월은 전기적 요인(약 43%)이 부주의를 넘어서고, 3~4월은 부주의가 60%를 넘습니다.
3. **시간**: 낮에는 부주의, 새벽에는 전기적 요인과 기타(방화 등)의 비중이 커집니다.
4. **피해**: 전기 화재의 재산피해 중앙값은 부주의의 약 4.7배이고, 기타(방화·가스 등)는 인명피해 발생률이 11.0%로 가장 높습니다.
5. **담배꽁초 하나가 전체 화재의 15.2%**입니다.

**3. 활용 방안**
- **맞춤 예방 홍보**: 여름에는 아파트·상가의 전기 안전 점검, 봄에는 논밭·쓰레기 태우기 금지와 담배꽁초 홍보처럼 시기·장소별로 집중할 원인을 정합니다.
- **화재 조사 보조**: 원인 조사 초기에 '이 장소·시간대에서 흔한 원인'을 우선 점검 목록으로 제시합니다.
- **원인 미상 화재 분석**: 미상 화재의 추정 원인 구성을 보고, 건물·차량 화재 조사 역량을 보강할 근거로 씁니다.

**4. 한계**
- **입력 정보가 적습니다.** 건물 노후도, 전기 설비 상태, 날씨(습도·기온) 같은 정보가 없어 같은 장소·시간의 화재를 구분하기 어렵습니다.
- '기타'는 방화·가스누출·교통사고 등 **성격이 다른 원인을 묶어서** 거의 맞히지 못합니다(재현율 8.8%).
- 원인 미상 화재의 추정은 '미상 화재도 원인을 아는 화재와 같은 패턴을 따른다'는 가정에 기댑니다.
- 모델이 확실하지 않으면 가장 흔한 '부주의'로 답하는 경향이 있어 전기·기계적 요인을 놓치는 경우가 많습니다.
- **하이퍼파라미터 튜닝은 하지 않았습니다.** 세 모델의 성능이 거의 같아서, 모델 설정보다 입력 정보가 성능을 제한하는 것으로 보입니다.

In [ ]:
# outputs 폴더에 저장된 PNG 파일 목록과 크기 확인
for f in sorted(os.listdir(OUT_DIR)):
    if f.endswith('.png'):
        print(f, f'{os.path.getsize(os.path.join(OUT_DIR, f)) / 1024:.0f} KB')